# 01 — CCLE Dataset: EDA, Outlier Detection, Feature Selection

Mirrors **Section 1 (CCLE Dataset)** of the reference project.

The DepMap files we have are the modern distribution of CCLE data:

| Reference file | Our file |
|---|---|
| `gene_expression.csv` | `OmicsExpressionProteinCodingGenesTPMLogp1.csv` |
| `mutations.csv` | `OmicsSomaticMutations.csv` |
| (metadata) | `Model.csv` |

**Pipeline of this notebook:**
1. Load + join → binary TP53 label
2. Train / validation / test split (everything downstream uses train only)
3. Per-gene stats across classes (mean, std, zero-expression %)
4. Pearson + Spearman with Benjamini–Hochberg correction
5. **Consensus outlier detection** (6 methods vote)
6. **Consensus feature selection** (correlation ∪ PCA ∪ TP53 target genes)
7. Save processed artifacts for the modeling notebooks

## 0. Setup

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 50)

RNG = 42
DATA_DIR = Path('data')   # edit this
OUT_DIR  = Path('processed'); OUT_DIR.mkdir(exist_ok=True)

MODEL_PATH = DATA_DIR / 'Model.csv'
MUT_PATH   = DATA_DIR / 'OmicsSomaticMutations.csv'
EXPR_PATH  = DATA_DIR / 'OmicsExpressionProteinCodingGenesTPMLogp1.csv'

## 1. Load + join → binary TP53 label

Filter mutations to TP53, left-join on expression, create binary `TP53_mut`.

In [ ]:
model = pd.read_csv(MODEL_PATH)
muts  = pd.read_csv(MUT_PATH, low_memory=False)
expr  = pd.read_csv(EXPR_PATH, index_col=0)
expr.columns = [c.split(' (')[0] for c in expr.columns]
expr.index.name = 'ModelID'

print(f'Model:      {model.shape}')
print(f'Mutations:  {muts.shape}')
print(f'Expression: {expr.shape}')

In [ ]:
# Permissive label: any TP53 entry in the called set → mutant.
# Document this choice in your report; you can tighten to LoF/hotspot later.
tp53_muts = muts[muts['HugoSymbol'] == 'TP53'].copy()
mut_ids   = set(tp53_muts['ModelID'])
called_ids = set(muts['ModelID'])

# Only cell lines that (a) appear in the mutation callset AND (b) have expression
common = sorted(called_ids & set(expr.index))

X_all = expr.loc[common].copy()
y_all = pd.Series([int(m in mut_ids) for m in common], index=common, name='TP53_mut')

print(f'Usable samples: {len(common):,}')
print(f'TP53 mutant:    {int(y_all.sum()):,} ({y_all.mean():.1%})')

# Also grab the mutation-type label for section-3 later (Ins/Del/SNV/Sub).
# We take the 'most severe' or first mutation per cell line for simplicity.
if 'VariantInfo' in tp53_muts.columns:
    mt_col = 'VariantInfo'
else:
    mt_col = 'VariantType'
mut_type = tp53_muts.groupby('ModelID')[mt_col].first()

## 2. Train / validation / test split

**All subsequent steps use `train` only.** Val + test are frozen until model evaluation.

In [ ]:
from sklearn.model_selection import train_test_split

X_temp, X_test, y_temp, y_test = train_test_split(
    X_all, y_all, test_size=0.15, stratify=y_all, random_state=RNG)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.15 / 0.85, stratify=y_temp, random_state=RNG)

print(f'Train: {X_train.shape}  mut%={y_train.mean():.1%}')
print(f'Val:   {X_val.shape}  mut%={y_val.mean():.1%}')
print(f'Test:  {X_test.shape}  mut%={y_test.mean():.1%}')

## 3. Per-gene statistics by class

For each gene, compare mean, std, and zero-expression rate between TP53-mut and WT. Reference finding: mutant lines have higher mean + std and lower zero rate.

In [ ]:
def gene_stats(X: pd.DataFrame, y: pd.Series, label: int) -> pd.DataFrame:
    Xg = X.loc[y == label]
    return pd.DataFrame({
        'mean': Xg.mean(),
        'std':  Xg.std(),
        'zero_pct': (Xg == 0).mean(),
    })

s_mut = gene_stats(X_train, y_train, 1)
s_wt  = gene_stats(X_train, y_train, 0)

cmp = pd.DataFrame({
    'mean_mut': s_mut['mean'], 'mean_wt': s_wt['mean'],
    'std_mut':  s_mut['std'],  'std_wt':  s_wt['std'],
    'zero_mut': s_mut['zero_pct'], 'zero_wt': s_wt['zero_pct'],
})
cmp['delta_mean'] = cmp['mean_mut'] - cmp['mean_wt']
cmp['delta_zero'] = cmp['zero_mut'] - cmp['zero_wt']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].scatter(cmp['mean_wt'], cmp['mean_mut'], s=2, alpha=0.3)
lim = [cmp[['mean_wt','mean_mut']].values.min(), cmp[['mean_wt','mean_mut']].values.max()]
axes[0].plot(lim, lim, 'r--', lw=1); axes[0].set_xlabel('mean WT'); axes[0].set_ylabel('mean MUT'); axes[0].set_title('Gene mean expression')
axes[1].scatter(cmp['std_wt'], cmp['std_mut'], s=2, alpha=0.3)
lim = [cmp[['std_wt','std_mut']].values.min(), cmp[['std_wt','std_mut']].values.max()]
axes[1].plot(lim, lim, 'r--', lw=1); axes[1].set_xlabel('std WT'); axes[1].set_ylabel('std MUT'); axes[1].set_title('Gene std')
axes[2].scatter(cmp['zero_wt'], cmp['zero_mut'], s=2, alpha=0.3)
axes[2].plot([0,1],[0,1], 'r--', lw=1); axes[2].set_xlabel('zero % WT'); axes[2].set_ylabel('zero % MUT'); axes[2].set_title('Gene zero-expression %')
plt.tight_layout()

print(f"Genes with higher mean in MUT: {(cmp['delta_mean'] > 0).mean():.1%}")
print(f"Genes with lower zero%  in MUT: {(cmp['delta_zero'] < 0).mean():.1%}")

## 4. Correlation with target: Pearson + Spearman + Benjamini–Hochberg

In [ ]:
from scipy.stats import pearsonr, spearmanr
from statsmodels.stats.multitest import multipletests

y_arr = y_train.values.astype(float)
Xv = X_train.values

# Vectorized Pearson via standardization
Xz = (Xv - Xv.mean(0)) / (Xv.std(0) + 1e-12)
yz = (y_arr - y_arr.mean()) / (y_arr.std() + 1e-12)
pearson_r = (Xz * yz[:, None]).mean(0)

# Pearson p-values via t-stat transform
n = len(y_arr)
t_stat = pearson_r * np.sqrt((n - 2) / (1 - pearson_r**2 + 1e-12))
pearson_p = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=n-2))

# Spearman on the matrix (can be slow — subsample if needed)
spearman_r, spearman_p = spearmanr(Xv, y_arr)
spearman_r = spearman_r[:-1, -1]
spearman_p = spearman_p[:-1, -1]

# Benjamini–Hochberg
_, pearson_p_bh, _, _  = multipletests(pearson_p,  method='fdr_bh')
_, spearman_p_bh, _, _ = multipletests(spearman_p, method='fdr_bh')

corr = pd.DataFrame({
    'pearson_r':  pearson_r,  'pearson_p_bh':  pearson_p_bh,
    'spearman_r': spearman_r, 'spearman_p_bh': spearman_p_bh,
}, index=X_train.columns)

print(corr.describe())

In [ ]:
# Reference finding: Pearson and Spearman should be nearly identical → relationships are mostly linear
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(corr['pearson_r'].abs(), corr['spearman_r'].abs(), s=2, alpha=0.2)
ax.plot([0,1],[0,1],'r--',lw=1)
ax.set_xlabel('|Pearson r|'); ax.set_ylabel('|Spearman r|')
ax.set_title('Linear vs monotonic correlation (per gene)')
plt.tight_layout()

agreement = np.corrcoef(corr['pearson_r'].abs(), corr['spearman_r'].abs())[0,1]
print(f'Correlation between |Pearson| and |Spearman|: {agreement:.3f}')

## 5. Consensus outlier detection

Six independent detectors vote. Sample flagged if ≥4 votes = **sure outlier** (dropped); ≥3 = **possible outlier** (kept, tracked). Contamination assumption: 5%.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler

CONTAM = 0.05
votes = pd.DataFrame(index=X_train.index)

# --- 1. Z-score: per gene, flag value if |z|>3.5; sample flagged if #flags > 95th pct
Z = (X_train - X_train.mean()) / (X_train.std() + 1e-12)
gene_flags = (Z.abs() > 3.5).sum(axis=1)
thr = np.quantile(gene_flags, 0.95)
votes['zscore'] = (gene_flags > thr).astype(int)

# --- 2. PCA + Mahalanobis + chi-squared
Xs = StandardScaler().fit_transform(X_train.values)
pcs = PCA(n_components=50, random_state=RNG).fit_transform(Xs)
cov = np.cov(pcs, rowvar=False)
inv = np.linalg.pinv(cov)
mu  = pcs.mean(axis=0)
d2  = np.einsum('ij,jk,ik->i', pcs-mu, inv, pcs-mu)
chi2_thr = stats.chi2.ppf(1 - CONTAM, df=pcs.shape[1])
votes['pca_mahal'] = (d2 > chi2_thr).astype(int)

# --- 3. Isolation Forest
iso = IsolationForest(contamination=CONTAM, random_state=RNG, n_jobs=-1).fit(Xs)
votes['iforest'] = (iso.predict(Xs) == -1).astype(int)

# --- 4. Local Outlier Factor
lof = LocalOutlierFactor(n_neighbors=20, contamination=CONTAM)
votes['lof'] = (lof.fit_predict(Xs) == -1).astype(int)

# --- 5. Zero-expression %: flag samples with > 95% zero genes
zero_pct = (X_train == 0).mean(axis=1)
votes['zero_pct'] = (zero_pct > 0.95).astype(int)

# --- 6. Total expression: flag samples outside the 5-95% range of sum
tot = X_train.sum(axis=1)
lo, hi = np.quantile(tot, [0.05, 0.95])
votes['total_expr'] = ((tot < lo) | (tot > hi)).astype(int)

votes['n_votes'] = votes.sum(axis=1)
print('Votes per method (fraction flagged):')
print(votes.drop(columns='n_votes').mean().round(3))
print('\nSamples by vote count:')
print(votes['n_votes'].value_counts().sort_index())

In [ ]:
sure_outliers     = votes.index[votes['n_votes'] >= 4]
possible_outliers = votes.index[votes['n_votes'] == 3]

print(f'Sure outliers (≥4 votes):     {len(sure_outliers)}')
print(f'Possible outliers (=3 votes): {len(possible_outliers)}')

# Drop only the sure ones
X_train_clean = X_train.drop(index=sure_outliers)
y_train_clean = y_train.drop(index=sure_outliers)
print(f'\nAfter cleaning: {X_train_clean.shape[0]} samples (from {X_train.shape[0]})')

## 6. Consensus feature selection

Union of three sets (recomputed on the cleaned train set):
- **Correlation:** |Pearson r| > 0.2 **and** BH-adjusted p ≤ 0.05
- **PCA support:** genes whose cumulative loading contributes to the first 80% of variance (we keep the top-loading genes across those PCs)
- **TP53 target genes:** curated list of canonical p53 targets

In [ ]:
# Recompute correlations on cleaned train
y_c = y_train_clean.values.astype(float)
Xc  = X_train_clean.values
Xcz = (Xc - Xc.mean(0)) / (Xc.std(0) + 1e-12)
ycz = (y_c - y_c.mean()) / (y_c.std() + 1e-12)
r   = (Xcz * ycz[:, None]).mean(0)
n   = len(y_c)
t   = r * np.sqrt((n - 2) / (1 - r**2 + 1e-12))
p   = 2 * (1 - stats.t.cdf(np.abs(t), df=n-2))
_, p_bh, _, _ = multipletests(p, method='fdr_bh')

# --- Set A: correlation-selected
set_corr = set(X_train_clean.columns[(np.abs(r) > 0.2) & (p_bh <= 0.05)])
print(f'Correlation-selected genes: {len(set_corr)}')

In [ ]:
# --- Set B: PCA 80% variance support
scaler_fs = StandardScaler().fit(Xc)
pca = PCA(n_components=0.80, random_state=RNG).fit(scaler_fs.transform(Xc))
# Take the union of the top-K loadings from each kept PC, with K scaled by PC count
top_k_per_pc = max(50, 3000 // pca.n_components_)
pca_genes = set()
for comp in pca.components_:
    idx = np.argsort(np.abs(comp))[-top_k_per_pc:]
    pca_genes.update(X_train_clean.columns[idx])
print(f'PCA components kept: {pca.n_components_} (80% var), genes: {len(pca_genes)}')

In [ ]:
# --- Set C: canonical TP53 target genes (curated)
# Replace this with the authoritative list from https://tp53.cancer.gov/target_genes if you can download it.
TP53_TARGETS = {
    'MDM2','CDKN1A','BAX','BBC3','PMAIP1','GADD45A','GADD45B','SESN1','SESN2','SESN3',
    'DDB2','XPC','POLH','BTG2','RPS27L','EDA2R','ZMAT3','TP53I3','TP53INP1','TP53AIP1',
    'GDF15','CCNG1','CCNG2','ATF3','RRAD','FDXR','TIGAR','FAS','PERP','RRM2B','SERPINE1',
    'MDM4','PIDD1','APAF1','BID','BAK1','CASP6','PTEN','IGFBP3','LIF','LRDD','NDRG1',
    'NOXA','PLK3','PLK2','SFN','SIAH1','THBS1','TNFRSF10B','WIG1','FOXO3','DRAM1',
}
set_targets = set(X_train_clean.columns) & TP53_TARGETS
print(f'TP53 target genes present: {len(set_targets)}')
print(f'Already picked by correlation: {len(set_targets & set_corr)}')

In [ ]:
# Union of the three sets
selected = sorted(set_corr | pca_genes | set_targets)
print(f'Total selected features: {len(selected)}  (from {X_train_clean.shape[1]})')

# Venn-like overlap counts
from itertools import combinations
sets = {'corr': set_corr, 'pca': pca_genes, 'targets': set_targets}
for k, v in sets.items():
    print(f'  only in {k}: {len(v - (set().union(*[s for kk,s in sets.items() if kk!=k])))}')
for a,b in combinations(sets, 2):
    print(f'  {a} ∩ {b}: {len(sets[a] & sets[b])}')
print(f'  all three:   {len(set_corr & pca_genes & set_targets)}')

## 7. Normality quick-check

Reference notes no preprocessing was needed because the data is log-transformed upstream (log2(TPM+1)). We sanity-check that here.

In [ ]:
from scipy.stats import shapiro
sample_genes = np.random.default_rng(RNG).choice(selected, size=min(30, len(selected)), replace=False)
p_values = []
for g in sample_genes:
    # Shapiro works on <=5000 samples; we're fine
    _, p = shapiro(X_train_clean[g].sample(min(500, len(X_train_clean)), random_state=RNG))
    p_values.append(p)
print(f'Shapiro p > 0.05 (fail to reject normality): {sum(p>0.05 for p in p_values)}/{len(p_values)} genes')
print('(Not a strict test — gene expression is rarely perfectly normal — but log-space is close enough for most linear methods.)')

## 8. Save processed artifacts for downstream notebooks

Next notebooks will load these directly — no need to re-run EDA.

In [ ]:
# Keep only selected features, apply to all three splits
X_train_final = X_train_clean[selected]
X_val_final   = X_val[selected]
X_test_final  = X_test[selected]

X_train_final.to_parquet(OUT_DIR / 'ccle_X_train.parquet')
X_val_final.to_parquet(  OUT_DIR / 'ccle_X_val.parquet')
X_test_final.to_parquet( OUT_DIR / 'ccle_X_test.parquet')

y_train_clean.to_frame().to_parquet(OUT_DIR / 'ccle_y_train.parquet')
y_val.to_frame().to_parquet(        OUT_DIR / 'ccle_y_val.parquet')
y_test.to_frame().to_parquet(       OUT_DIR / 'ccle_y_test.parquet')

# Full (unfiltered, uncleaned) expression matrix — needed for differential expression (Section 5)
X_train.to_parquet(OUT_DIR / 'ccle_X_train_full.parquet')

# Mutation type labels for multiclass task
mut_type.to_frame('mut_type').to_parquet(OUT_DIR / 'ccle_mut_type.parquet')

# Diagnostic tables
votes.to_parquet(OUT_DIR / 'ccle_outlier_votes.parquet')
corr.to_parquet( OUT_DIR / 'ccle_corr_pre_cleaning.parquet')

print(f'Saved to {OUT_DIR.resolve()}:')
for f in sorted(OUT_DIR.iterdir()):
    print(f'  {f.name:35s} {f.stat().st_size/1e6:6.2f} MB')

## What to put in the report for this section

1. **Data summary** — sizes, class balance, splitting strategy.
2. **Per-class gene statistics** — confirm or refute the reference's finding that mutant lines show higher mean / std and lower zero rate.
3. **Correlation linearity** — |Pearson| vs |Spearman| agreement → confirms relationships are mostly linear.
4. **Outlier counts** — how many samples each method flagged; how many had ≥4 votes (dropped).
5. **Feature selection breakdown** — sizes of the three sets, their intersection, and how many canonical TP53 targets the correlation method independently found (a nice sanity signal).

### Next notebooks
- `02_ccle_binary.ipynb` — DT / RF / KNN / MLP / XGBoost + stacking on TP53 binary task
- `03_ccle_multiclass.ipynb` — mutation-type classification with SMOTE + ANOVA/MI feature selection
- `04_tcga_download.ipynb` — GDC transfer tool, then parallel pipeline
- `05_mutation_analysis.ipynb` — Mann-Whitney + log2FC volcano plot, per-mutation-type comparison